# Module 8: Model-Based RL — Monte Carlo Tree Search for Tic-Tac-Toe

## Learning Objectives
By completing this notebook, you will:
1. Understand why search-based planning differs from learned-model planning
2. Implement all four phases of MCTS: selection (UCT), expansion, simulation, and backpropagation
3. Build a complete game tree for Tic-Tac-Toe and evaluate move quality
4. Measure how win rate against a random opponent improves with more simulations

## Prerequisites
- `01_dyna_q.ipynb` (Module 8) — model-based RL concepts
- Module 2: Monte Carlo methods — expected value estimation from rollouts

## Estimated Time: 15 minutes

---

## Search vs. Learning

Dyna-Q *learns* an approximate model from data. MCTS assumes access to a **perfect simulator** — a function that given a state and action returns the exact next state. This is true for board games, turn-based puzzles, and any environment with known rules.

MCTS builds a search tree by running **many simulated games** from the current position. Each simulation consists of:
1. **Selection** — traverse the tree to a promising leaf using UCT (Upper Confidence bounds for Trees)
2. **Expansion** — add a new child node
3. **Simulation** — play randomly to a terminal state
4. **Backpropagation** — update win/visit counts up the tree

After $N$ simulations, pick the action with the most visits.

In [ ]:
learning_objectives([
    "`01_dyna_q.ipynb` (Module 8) — model-based RL concepts",
    "Module 2: Monte Carlo methods — expected value estimation from rollouts",
    "## Search vs. Learning",
    "an approximate model from data. MCTS assumes access to a **perfect simulator** — a function that given a state and action returns the exact next state. This is true for board games, turn-based puzzles, and any environment with known rules.",
    "from the current position. Each simulation consists of:",
    "— traverse the tree to a promising leaf using UCT (Upper Confidence bounds for Trees)",
    "— add a new child node",
    "— play randomly to a terminal state",
    "— update win/visit counts up the tree"
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import dependencies
# Key Concept: MCTS needs only numpy and math — no learning, no gradients

import numpy as np
import matplotlib.pyplot as plt
import math
from copy import deepcopy
from typing import Optional, List
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print("MCTS Tic-Tac-Toe: Monte Carlo Tree Search from scratch")

## 1. Tic-Tac-Toe Game Engine

The game state is a 3x3 board represented as a flat numpy array:
- 0 = empty, 1 = X (current player), -1 = O (opponent)

We always represent the board from the perspective of the **current player**, so player 1 is always X.

In [ ]:
# Purpose: Implement the Tic-Tac-Toe game logic
# Key Concept: Immutable game state enables safe tree search (each node is a separate board)

class TicTacToe:
    """
    Tic-Tac-Toe game engine.

    Board: 3x3 grid, values {0: empty, 1: X, -1: O}
    Player 1 is always X, player -1 is always O.
    MCTS uses player perspective flipping for generality.
    """

    WIN_LINES = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],   # rows
        [0, 3, 6], [1, 4, 7], [2, 5, 8],   # cols
        [0, 4, 8], [2, 4, 6]               # diagonals
    ]

    def __init__(self, board: Optional[np.ndarray] = None, current_player: int = 1):
        self.board = board if board is not None else np.zeros(9, dtype=int)
        self.current_player = current_player  # 1 or -1

    def legal_moves(self) -> List[int]:
        """Return indices of empty cells."""
        return [i for i in range(9) if self.board[i] == 0]

    def make_move(self, cell: int) -> 'TicTacToe':
        """
        Return a new game state after placing current player's mark in cell.

        Parameters
        ----------
        cell : int
            Cell index (0-8).

        Returns
        -------
        TicTacToe
            New game state with next player to move.
        """
        new_board = self.board.copy()
        new_board[cell] = self.current_player
        return TicTacToe(new_board, -self.current_player)

    def winner(self) -> Optional[int]:
        """
        Return the winning player (1 or -1), 0 for draw, None if game not over.
        """
        for line in self.WIN_LINES:
            s = self.board[line].sum()
            if s == 3:
                return 1
            if s == -3:
                return -1
        if len(self.legal_moves()) == 0:
            return 0  # Draw
        return None  # Game ongoing

    def is_terminal(self) -> bool:
        return self.winner() is not None

    def render(self):
        """Print the board."""
        symbols = {0: '.', 1: 'X', -1: 'O'}
        for row in range(3):
            print(' '.join([symbols[self.board[row * 3 + col]] for col in range(3)]))
        print()


# Verify game logic
game = TicTacToe()
# Simulate X winning: X takes 0,1,2 (top row)
for move in [0, 3, 1, 4, 2]:  # X:0,1,2 O:3,4
    game = game.make_move(move)
game.render()
print(f"Winner: {game.winner()}  (1=X, -1=O, 0=draw, None=ongoing)")
print(f"Legal moves: {game.legal_moves()}")

## 2. MCTS Node

Each node in the search tree represents a game state. It stores:
- `wins`: total wins accumulated from simulations passing through this node
- `visits`: total visits
- `children`: child nodes (one per legal move from this state)
- `untried_moves`: legal moves not yet expanded

The **UCT score** balances exploitation (high win rate) with exploration (rarely visited nodes):

$$\text{UCT}(n) = \frac{w_n}{v_n} + c \sqrt{\frac{\ln v_{\text{parent}}}{v_n}}$$

where $c = \sqrt{2}$ is the exploration constant.

In [ ]:
# Purpose: Implement the MCTS tree node
# Key Concept: UCT balances exploitation and exploration — the same tradeoff as epsilon-greedy

class MCTSNode:
    """
    Node in the Monte Carlo Search Tree.

    Parameters
    ----------
    state : TicTacToe
        Game state at this node.
    parent : MCTSNode or None
        Parent node (None for root).
    move : int or None
        Move that led to this node from parent.
    """

    def __init__(
        self,
        state: TicTacToe,
        parent: Optional['MCTSNode'] = None,
        move: Optional[int] = None
    ):
        self.state = state
        self.parent = parent
        self.move = move

        self.wins = 0.0
        self.visits = 0
        self.children: List['MCTSNode'] = []
        self.untried_moves = state.legal_moves()

    def is_fully_expanded(self) -> bool:
        """True if all legal moves have been tried at least once."""
        return len(self.untried_moves) == 0

    def uct_score(self, exploration_constant: float = math.sqrt(2)) -> float:
        """
        Compute UCT score for this node from parent's perspective.

        Parameters
        ----------
        exploration_constant : float
            Exploration weight c in UCT formula.

        Returns
        -------
        float
            UCT score (higher = more promising to explore).
        """
        if self.visits == 0:
            return float('inf')  # Unvisited nodes have highest priority

        exploitation = self.wins / self.visits
        exploration = exploration_constant * math.sqrt(
            math.log(self.parent.visits) / self.visits
        )
        return exploitation + exploration

    def best_child(self, exploration_constant: float = math.sqrt(2)) -> 'MCTSNode':
        """Return child with highest UCT score."""
        return max(self.children, key=lambda c: c.uct_score(exploration_constant))

    def expand(self) -> 'MCTSNode':
        """
        Add one new child for a randomly chosen untried move.

        Returns
        -------
        MCTSNode
            The newly created child node.
        """
        move = self.untried_moves.pop(np.random.randint(len(self.untried_moves)))
        new_state = self.state.make_move(move)
        child = MCTSNode(new_state, parent=self, move=move)
        self.children.append(child)
        return child


# Verify
root = MCTSNode(TicTacToe())
print(f"Root untried moves: {root.untried_moves}")
child = root.expand()
print(f"After expand: root has {len(root.children)} child, untried: {len(root.untried_moves)}")

## 3. The Four MCTS Phases

**Selection**: Walk down the tree choosing the child with highest UCT score until reaching an unexpanded node or terminal state.

**Expansion**: Add one new child to the selected node.

**Simulation**: Play random moves from the expanded node until the game ends.

**Backpropagation**: Walk back up the tree incrementing visits and wins (from the perspective of the node's player).

In [ ]:
# Purpose: Implement the four MCTS phases as standalone functions
# Key Concept: Each phase is clearly separable — makes debugging and modification easy

def select(node: MCTSNode) -> MCTSNode:
    """
    Phase 1: Traverse the tree from root to a leaf using UCT.

    Keeps selecting best_child() until reaching a node that:
    - Is terminal, OR
    - Has untried moves (not fully expanded)

    Returns
    -------
    MCTSNode
        The selected leaf node.
    """
    current = node
    while not current.state.is_terminal() and current.is_fully_expanded():
        current = current.best_child()
    return current


def simulate(state: TicTacToe) -> int:
    """
    Phase 3: Play random moves from `state` until terminal.

    Returns
    -------
    int
        Winner (1, -1) or draw (0).
    """
    current = state
    while not current.is_terminal():
        moves = current.legal_moves()
        move = moves[np.random.randint(len(moves))]
        current = current.make_move(move)
    return current.winner()


def backpropagate(node: MCTSNode, result: int, root_player: int):
    """
    Phase 4: Walk from node back to root, updating wins and visits.

    Win = +1 if result matches root_player, -1 for opponent win, +0.5 for draw.
    Each node updates from the perspective of the player who *moved into* it.

    Parameters
    ----------
    node : MCTSNode
        Starting node (leaf).
    result : int
        Game outcome: 1, -1, or 0.
    root_player : int
        The player MCTS is searching for (1 or -1).
    """
    current = node
    while current is not None:
        current.visits += 1
        # The player who moved INTO this node is the parent's current_player
        node_player = -current.state.current_player  # player who just moved
        if result == node_player:
            current.wins += 1.0
        elif result == 0:
            current.wins += 0.5  # draw
        current = current.parent


def mcts_search(
    state: TicTacToe,
    n_simulations: int = 100,
    exploration_c: float = math.sqrt(2)
) -> int:
    """
    Run MCTS from the given state and return the best move.

    Parameters
    ----------
    state : TicTacToe
        Current game state.
    n_simulations : int
        Number of MCTS simulations to run.
    exploration_c : float
        UCT exploration constant.

    Returns
    -------
    int
        Best move (cell index 0-8).
    """
    root = MCTSNode(state)
    root_player = state.current_player

    for _ in range(n_simulations):
        # Phase 1: Selection
        leaf = select(root)

        # Phase 2: Expansion (if not terminal)
        if not leaf.state.is_terminal():
            leaf = leaf.expand()

        # Phase 3: Simulation
        result = simulate(leaf.state)

        # Phase 4: Backpropagation
        backpropagate(leaf, result, root_player)

    # Choose move with most visits (most robust estimate)
    best = max(root.children, key=lambda c: c.visits)
    return best.move


# Quick test: from empty board, MCTS should produce a valid move
np.random.seed(SEED)
test_game = TicTacToe()
move = mcts_search(test_game, n_simulations=50)
print(f"MCTS best move from empty board (50 sims): cell {move}")
print(f"Legal moves: {test_game.legal_moves()}")
assert move in test_game.legal_moves(), "MCTS must return a legal move"

## 4. Win Rate vs. Number of Simulations

We play MCTS against a random opponent and measure win rate as a function of simulation count. This reveals the information value of each simulation.

In [ ]:
# Purpose: Evaluate MCTS win rate against a random opponent
# Key Concept: More simulations = better tree coverage = better move selection

def play_game_mcts_vs_random(
    n_simulations: int,
    mcts_player: int = 1,
    seed: Optional[int] = None
) -> int:
    """
    Play one game: MCTS as `mcts_player` vs random opponent.

    Returns
    -------
    int
        1 if MCTS wins, -1 if random wins, 0 for draw.
    """
    if seed is not None:
        np.random.seed(seed)

    game = TicTacToe()

    while not game.is_terminal():
        if game.current_player == mcts_player:
            move = mcts_search(game, n_simulations)
        else:
            moves = game.legal_moves()
            move = moves[np.random.randint(len(moves))]

        game = game.make_move(move)

    return game.winner()


def evaluate_mcts(
    n_simulations: int,
    n_games: int = 100,
    seed: int = SEED
) -> dict:
    """
    Evaluate MCTS win/draw/loss rates over n_games against random opponent.

    Returns
    -------
    dict with keys: 'win_rate', 'draw_rate', 'loss_rate'
    """
    np.random.seed(seed)
    outcomes = [play_game_mcts_vs_random(n_simulations) for _ in range(n_games)]
    wins = outcomes.count(1)
    draws = outcomes.count(0)
    losses = outcomes.count(-1)
    return {
        'win_rate': wins / n_games,
        'draw_rate': draws / n_games,
        'loss_rate': losses / n_games
    }


# Evaluate at several simulation counts
sim_counts = [1, 5, 10, 50, 100, 200, 500]
eval_results = {}

for n_sims in sim_counts:
    print(f"Evaluating n_simulations={n_sims:4d}... ", end='', flush=True)
    eval_results[n_sims] = evaluate_mcts(n_sims, n_games=80)
    wr = eval_results[n_sims]['win_rate']
    print(f"win rate: {wr:.2f}")

## 5. Visualizing Results

In [ ]:
# Purpose: Visualize win rate vs. simulation count and a sample game tree
# Key Concept: The win rate plateau shows the point of diminishing returns for computation

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: Win/draw/loss rates vs. simulations ---
ax = axes[0]
x = list(eval_results.keys())
win_rates = [eval_results[n]['win_rate'] for n in x]
draw_rates = [eval_results[n]['draw_rate'] for n in x]
loss_rates = [eval_results[n]['loss_rate'] for n in x]

ax.stackplot(x, win_rates, draw_rates, loss_rates,
             labels=['Win', 'Draw', 'Loss'],
             colors=['#2ca02c', '#ff7f0e', '#d62728'],
             alpha=0.8)
ax.set_xscale('log')
ax.set_xlabel('Number of MCTS Simulations (log scale)', fontsize=12)
ax.set_ylabel('Game Outcome Rate', fontsize=12)
ax.set_title('MCTS vs. Random Opponent: Win Rate Improvement', fontsize=12)
ax.legend(loc='center right', fontsize=10)
ax.set_xlim([1, max(x)])
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

# --- Plot 2: Node visits in MCTS tree (from empty board, 200 sims) ---
ax = axes[1]
np.random.seed(SEED)

# Run MCTS and get root's children visit counts
root_state = TicTacToe()
root_node = MCTSNode(root_state)

for _ in range(300):
    leaf = select(root_node)
    if not leaf.state.is_terminal():
        leaf = leaf.expand()
    result = simulate(leaf.state)
    backpropagate(leaf, result, 1)

# Build visit count grid
visit_grid = np.zeros(9)
win_grid = np.zeros(9)
for child in root_node.children:
    visit_grid[child.move] = child.visits
    win_grid[child.move] = child.wins / child.visits if child.visits > 0 else 0

im = ax.imshow(visit_grid.reshape(3, 3), cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label='Visit count')

# Annotate with visit counts and win rates
for i in range(3):
    for j in range(3):
        idx = i * 3 + j
        v = int(visit_grid[idx])
        w = win_grid[idx]
        ax.text(j, i, f'{v}\n({w:.2f})',
                ha='center', va='center', fontsize=9,
                color='white' if v > visit_grid.max() * 0.5 else 'black')

ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_title('MCTS Visit Counts after 300 Sims\n(visits | win rate)', fontsize=12)
ax.set_xlabel('Column')
ax.set_ylabel('Row')

plt.tight_layout()
plt.savefig('mcts_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to mcts_analysis.png")

## 6. Tracing a Sample Game

In [ ]:
# Purpose: Play and display a complete game between MCTS and random opponent
# Key Concept: Watching MCTS play reveals its strategy and corner/center preferences

def play_and_display(n_simulations: int = 200, seed: int = SEED):
    """Play one game and display each move."""
    np.random.seed(seed)
    game = TicTacToe()
    print(f"MCTS (X, player 1) vs Random (O, player -1) | Sims: {n_simulations}")
    print("Initial board:")
    game.render()

    move_num = 0
    while not game.is_terminal():
        move_num += 1
        if game.current_player == 1:
            move = mcts_search(game, n_simulations)
            player_name = "MCTS (X)"
        else:
            moves = game.legal_moves()
            move = moves[np.random.randint(len(moves))]
            player_name = "Random (O)"

        game = game.make_move(move)
        print(f"Move {move_num}: {player_name} plays cell {move}")
        game.render()

    w = game.winner()
    if w == 1:
        print("Result: MCTS wins!")
    elif w == -1:
        print("Result: Random opponent wins!")
    else:
        print("Result: Draw")


play_and_display(n_simulations=200)

## Exercise 8.3: UCT Exploration Constant

**Task:** The exploration constant $c$ in UCT controls the exploration-exploitation tradeoff. Higher $c$ means more exploration of less-visited nodes; lower $c$ means more exploitation of high-win-rate nodes.

Evaluate MCTS with `exploration_c` in `{0.1, math.sqrt(2), 2.0, 5.0}` using 100 simulations per move, 60 games each. Report win rates and identify which `c` performs best.

<details>
<summary>Hint 1</summary>
Modify `evaluate_mcts` to accept an `exploration_c` argument, or write a new evaluation function that passes `exploration_c` to `mcts_search`.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
In `mcts_search`, the `best_child` call uses `exploration_c`. Make `mcts_search` accept an `exploration_c` parameter and thread it through.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def mcts_search_with_c(
    state: TicTacToe,
    n_simulations: int = 100,
    exploration_c: float = math.sqrt(2)
) -> int:
    """
    MCTS search with configurable exploration constant.
    Same as mcts_search but passes exploration_c to best_child.

    Returns
    -------
    int
        Best move cell index.
    """
    # YOUR CODE HERE
    pass


exploration_values = [0.1, math.sqrt(2), 2.0, 5.0]
exploration_results = {}  # Map c -> win_rate

# Fill exploration_results and print a table

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_8_3():
    # Test mcts_search_with_c returns a legal move
    np.random.seed(0)
    g = TicTacToe()
    for c in [0.1, math.sqrt(2), 5.0]:
        move = mcts_search_with_c(g, n_simulations=20, exploration_c=c)
        assert move is not None, f"mcts_search_with_c returned None for c={c}"
        assert move in g.legal_moves(), \
            f"c={c}: returned illegal move {move}, legal={g.legal_moves()}"

    assert len(exploration_results) == 4, \
        f"Expected 4 exploration experiments, got {len(exploration_results)}"
    for c in exploration_values:
        assert c in exploration_results, f"Missing result for c={c}"
        wr = exploration_results[c]
        assert 0.0 <= wr <= 1.0, f"Win rate must be in [0,1], got {wr} for c={c}"

    print("Exercise 8.3 passed! UCT exploration constant analysis complete.")

test_exercise_8_3()

## Exercise 8.4: MCTS vs MCTS

**Task:** Run a tournament between MCTS with different simulation budgets. Play MCTS(100 sims) vs MCTS(10 sims) for 40 games. Report the win rate of the stronger player.

**Expected observation:** More simulations should win most games — but the margin should decrease as both agents improve.

<details>
<summary>Hint 1</summary>
Write `play_game_mcts_vs_mcts(n_sims_1, n_sims_2)` where player 1 uses n_sims_1 simulations and player -1 uses n_sims_2.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def play_game_mcts_vs_mcts(n_sims_player1: int, n_sims_player2: int) -> int:
    """
    Play one game: MCTS(n_sims_player1) as X vs MCTS(n_sims_player2) as O.

    Returns
    -------
    int
        Winner: 1 (X wins), -1 (O wins), 0 (draw).
    """
    # YOUR CODE HERE
    pass


n_games_tournament = 40
tournament_win_rate = None  # Fraction of games player 1 (100 sims) wins

# Run the tournament and report results

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_8_4():
    # Test play_game_mcts_vs_mcts returns a valid outcome
    np.random.seed(0)
    result = play_game_mcts_vs_mcts(10, 10)
    assert result is not None, "play_game_mcts_vs_mcts must return a value"
    assert result in [-1, 0, 1], \
        f"Result must be -1, 0, or 1, got {result}"

    assert tournament_win_rate is not None, \
        "tournament_win_rate must be set (fraction of wins for 100-sim player)"
    assert 0.0 <= tournament_win_rate <= 1.0, \
        f"Win rate must be in [0, 1], got {tournament_win_rate}"

    print(f"Exercise 8.4 passed! MCTS(100) vs MCTS(10): win rate = {tournament_win_rate:.2f}")

test_exercise_8_4()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **MCTS requires no learning** — it uses a perfect simulator to plan by searching the game tree. This makes it exact in deterministic games but inapplicable when the environment model is unknown.

2. **UCT is the key innovation**: the $\sqrt{\ln N_{parent} / N_{child}}$ exploration term ensures every node is eventually visited while preferring high-value regions. It's the same exploitation-exploration tradeoff as UCB in multi-armed bandits.

3. **More simulations = better move quality**, but with diminishing returns. The win rate curve against a random opponent typically plateaus after ~100-200 simulations for Tic-Tac-Toe.

4. **AlphaGo and AlphaZero** combine MCTS with neural networks: the network provides a prior over moves (replacing random simulation) and a value estimate (replacing full playouts). This is the frontier of model-based RL.

5. **The exploration constant $c = \sqrt{2}$** is theoretically motivated but often tuned in practice. Games with high variance (long random simulations) benefit from lower $c$.

## What's Next

Module 9 explores the **frontiers** of RL: offline learning (learning from a fixed dataset without environment interaction) and RL for sequential decision-making in financial markets.

## Additional Resources

- Browne et al. (2012): "A Survey of Monte Carlo Tree Search Methods" — comprehensive overview
- Silver et al. (2016): "Mastering the Game of Go with Deep Neural Networks and Tree Search" — AlphaGo
- Silver et al. (2017): "Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm" — AlphaZero

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])